<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/05_Analyse_Radiographies_Thoraciques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Analyse de Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

- Utiliser TorchXRayVision pour détecter 18 pathologies thoraciques
- Charger et analyser des radiographies
- Comprendre les limites: qualité d'image et pathologies hors distribution

## Pathologies Détectables (18)

Atélectasie, Cardiomégalie, Épanchement, Infiltration, Masse, Nodule, Pneumonie, Pneumothorax, Consolidation, Œdème, Emphysème, Fibrose, Épaississement pleural, Hernie, Opacité pulmonaire, Lésion pleurale, Fracture, Dispositif médical

## 1. Installation

In [ ]:
!pip install -q torchxrayvision torch torchvision scikit-image pillow matplotlib

In [ ]:
import torchxrayvision as xrv
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import skimage.io
import skimage.transform
import skimage.util
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Chargement du Modèle

In [ ]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model = model.to(device)
model.eval()

print(f"Pathologies détectables ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies, 1):
    print(f"{i:2d}. {p}")

## 3. Fonctions Utilitaires

In [ ]:
# Transformations standard
transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])

def prepare_image(img_input, from_file=True):
    """
    Prépare une image pour le modèle.
    
    Args:
        img_input: chemin du fichier (si from_file=True) ou array numpy
        from_file: si True, charge depuis fichier, sinon utilise l'array directement
    
    Returns:
        tensor: image préparée pour le modèle
    """
    if from_file:
        img = skimage.io.imread(img_input)
    else:
        img = img_input
    
    # Normaliser
    img = xrv.datasets.normalize(img, 255)
    
    # Convertir en grayscale si nécessaire
    if len(img.shape) == 3:
        img = img.mean(2)
    img = img[None, ...]
    
    # Appliquer transformations
    img = transform(img)
    
    # Convertir en tensor
    img_tensor = torch.from_numpy(img).to(device)
    return img_tensor


def predict(img_tensor, model):
    """
    Prédit les pathologies sur une radiographie.
    
    Args:
        img_tensor: tensor de l'image préparée
        model: modèle TorchXRayVision
    
    Returns:
        probs: probabilités pour chaque pathologie
        indices: indices triés par probabilité décroissante
    """
    with torch.no_grad():
        pred = model(img_tensor[None,...])
        probs = torch.sigmoid(pred).cpu().numpy()[0]
    
    indices = np.argsort(probs)[::-1]
    return probs, indices


def display_predictions(probs, indices, model, top_n=8):
    """
    Affiche les prédictions sous forme de texte.
    
    Args:
        probs: probabilités
        indices: indices triés
        model: modèle (pour les noms de pathologies)
        top_n: nombre de pathologies à afficher
    """
    print("\nRÉSULTATS")
    print("="*55)
    for idx in indices[:top_n]:
        pathology = model.pathologies[idx]
        prob = probs[idx]
        bar = '█' * int(prob * 30)
        print(f"{pathology:<25} {prob:>6.1%} {bar}")
    print("="*55)


def visualize_results(img_tensor, probs, indices, model, top_n=8):
    """
    Visualise l'image et les prédictions.
    
    Args:
        img_tensor: tensor de l'image
        probs: probabilités
        indices: indices triés
        model: modèle (pour les noms de pathologies)
        top_n: nombre de pathologies à afficher
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Image
    ax1.imshow(img_tensor[0].cpu(), cmap='gray')
    ax1.set_title("Radiographie")
    ax1.axis('off')
    
    # Barres
    top_indices = indices[:top_n]
    top_pathologies = [model.pathologies[i] for i in top_indices]
    top_probs = [probs[i] for i in top_indices]
    
    colors = ['red' if p > 0.5 else 'orange' if p > 0.3 else 'steelblue' for p in top_probs]
    ax2.barh(range(top_n), top_probs, color=colors)
    ax2.set_yticks(range(top_n))
    ax2.set_yticklabels(top_pathologies)
    ax2.set_xlabel('Probabilité')
    ax2.set_title('Top Pathologies')
    ax2.set_xlim(0, 1)
    ax2.axvline(x=0.5, color='red', linestyle='--', alpha=0.3, label='Seuil 50%')
    ax2.axvline(x=0.3, color='orange', linestyle='--', alpha=0.3, label='Seuil 30%')
    ax2.legend(loc='lower right', fontsize=8)
    ax2.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("Fonctions utilitaires chargées!")

## 4. Chargement d'Images

### Option A: Depuis un Dataset TorchXRayVision

In [ ]:
# Tenter de charger un dataset
dataset_img = None

try:
    dataset = xrv.datasets.NIH_Google_Dataset(
        imgpath="./images",
        csvpath="./nih_google.csv",
        transform=transform
    )
    
    # Charger une image
    sample = dataset[0]
    dataset_img = sample['img'].to(device)
    
    # Afficher
    plt.figure(figsize=(6, 6))
    plt.imshow(dataset_img[0].cpu(), cmap='gray')
    plt.title("Image du Dataset")
    plt.axis('off')
    plt.show()
    
    print("Dataset chargé avec succès!")
    
except Exception as e:
    print(f"Dataset non disponible: {e}")
    print("Utilisez l'Option B (Upload) ci-dessous.")

### Option B: Upload de Votre Image

In [ ]:
# Upload (ne s'exécute que si dataset non chargé)
uploaded_img = None

if dataset_img is None:
    from google.colab import files
    
    uploaded = files.upload()
    
    if uploaded:
        img_path = list(uploaded.keys())[0]
        uploaded_img = prepare_image(img_path, from_file=True)
        
        # Afficher
        plt.figure(figsize=(6, 6))
        plt.imshow(uploaded_img[0].cpu(), cmap='gray')
        plt.title("Votre Image")
        plt.axis('off')
        plt.show()
        
        print(f"Image chargée: {img_path}")
else:
    print("Dataset déjà chargé, upload non nécessaire.")

### Sélection de l'Image à Analyser

In [ ]:
# Utiliser l'image disponible
if dataset_img is not None:
    main_img = dataset_img
    print("Utilisation de l'image du dataset")
elif uploaded_img is not None:
    main_img = uploaded_img
    print("Utilisation de l'image uploadée")
else:
    raise ValueError("Aucune image chargée. Exécutez Option A ou B.")

## 5. Analyse de l'Image Principale

In [ ]:
# Prédiction
probs, indices = predict(main_img, model)

# Affichage texte
display_predictions(probs, indices, model, top_n=10)

## 6. Visualisation

In [ ]:
visualize_results(main_img, probs, indices, model, top_n=8)

## 7. Impact des Distorsions d'Image

In [ ]:
def apply_distortions(img_tensor):
    """
    Applique différentes distorsions à une image.
    
    Args:
        img_tensor: tensor de l'image
    
    Returns:
        dict: {nom_distorsion: array_image}
    """
    img_array = (img_tensor[0].cpu().numpy() * 255).astype(np.uint8)
    
    distortions = {}
    distortions['Original'] = img_array
    
    # Rotation
    distortions['Rotation 15°'] = skimage.transform.rotate(
        img_array, 15, mode='edge', preserve_range=True
    ).astype(np.uint8)
    
    # Zoom
    h, w = img_array.shape
    crop_size = int(min(h, w) * 0.7)
    start = (h - crop_size) // 2
    cropped = img_array[start:start+crop_size, start:start+crop_size]
    distortions['Zoom'] = skimage.transform.resize(
        cropped, img_array.shape, preserve_range=True
    ).astype(np.uint8)
    
    # Bruit
    noisy = skimage.util.random_noise(img_array / 255.0, mode='gaussian', var=0.01)
    distortions['Bruit'] = (noisy * 255).astype(np.uint8)
    
    # Faible contraste
    mean_val = img_array.mean()
    low_contrast = ((img_array - mean_val) * 0.5 + mean_val).clip(0, 255)
    distortions['Faible Contraste'] = low_contrast.astype(np.uint8)
    
    return distortions


def analyze_distortions(distortions_dict, model):
    """
    Analyse chaque version distordue avec le modèle.
    
    Args:
        distortions_dict: dictionnaire {nom: image_array}
        model: modèle TorchXRayVision
    
    Returns:
        dict: {nom: {'image': array, 'predictions': [(patho, prob), ...]}}
    """
    results = {}
    
    for name, img_array in distortions_dict.items():
        # Préparer pour le modèle
        img_tensor = prepare_image(img_array, from_file=False)
        
        # Prédiction
        probs, indices = predict(img_tensor, model)
        
        # Top 3
        top_preds = [(model.pathologies[indices[i]], probs[indices[i]]) for i in range(3)]
        
        results[name] = {'image': img_array, 'predictions': top_preds}
    
    return results


# Appliquer distorsions
distortions = apply_distortions(main_img)
print(f"Créé {len(distortions)} versions")

# Analyser
distortion_results = analyze_distortions(distortions, model)
print("Analyse terminée!")

In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for idx, (name, data) in enumerate(distortion_results.items()):
    if idx >= 4:
        break
    
    axes[idx].imshow(data['image'], cmap='gray')
    
    top_pred = data['predictions'][0]
    title = f"{name}\n{top_pred[0]}: {top_pred[1]*100:.1f}%"
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# Tableau
print("\nIMPACT DES DISTORSIONS")
print("="*70)
for name, data in distortion_results.items():
    print(f"\n{name:20s}")
    for i, (patho, prob) in enumerate(data['predictions'], 1):
        print(f"  {i}. {patho:25s} {prob*100:5.1f}%")
print("\n" + "="*70)

## 8. Pathologies Out-of-Distribution (OOD)

### Exemple: COVID-19

Le modèle a été entraîné AVANT la pandémie COVID. Testons avec une vraie radiographie COVID.

**Dataset**: https://github.com/ieee8023/covid-chestxray-dataset

In [ ]:
# Télécharger une radiographie COVID réelle
!wget -q https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/master/images/01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg -O covid_xray.jpg

# Préparer l'image
covid_img = prepare_image('covid_xray.jpg', from_file=True)

# Afficher
plt.figure(figsize=(6, 6))
plt.imshow(covid_img[0].cpu(), cmap='gray')
plt.title("Radiographie COVID-19 (Dataset Réel)")
plt.axis('off')
plt.show()

print("Image COVID-19 téléchargée.")
print("Le modèle n'a JAMAIS vu de COVID pendant l'entraînement.")

In [ ]:
# Analyser l'image COVID
covid_probs, covid_indices = predict(covid_img, model)

print("\nPRÉDICTIONS POUR RADIOGRAPHIE COVID-19")
print("="*55)
print("Rappel: Le modèle N'A PAS été entraîné sur COVID!")
print("="*55)

display_predictions(covid_probs, covid_indices, model, top_n=8)

print("\nOBSERVATION:")
print("Le modèle mappe COVID vers des catégories connues:")
print("  - Pneumonia, Infiltration, Edema, Lung Opacity")
print("\nC'est typique d'une pathologie OUT-OF-DISTRIBUTION (OOD).")
print("Le modèle utilise les catégories les plus proches qu'il connaît.")

## 9. Points Clés

### Ce que vous avez appris:
- Charger et analyser des radiographies avec TorchXRayVision
- Utiliser des fonctions réutilisables pour preprocessing et prédictions
- Impact de la qualité d'image sur les prédictions
- Comportement avec pathologies OOD (COVID exemple)

### Applications:
- **Triage**: Prioriser cas urgents
- **Aide au diagnostic**: Détection rapide
- **Recherche**: Analyser grandes cohortes

### Limites importantes:
- **Qualité d'image**: Rotation, bruit, contraste affectent les résultats
- **Pathologies OOD**: Le modèle mappe vers catégories connues
- **Populations**: Entraîné sur certaines populations
- **Outil d'aide**: Ne remplace pas l'expertise médicale

### Bonnes pratiques de code:
- **Fonctions réutilisables**: `prepare_image()`, `predict()`, `display_predictions()`, `visualize_results()`
- **Variables distinctes**: `dataset_img`, `uploaded_img`, `main_img`, `covid_img` évitent les écrasements
- **Modularité**: Chaque fonction a une responsabilité claire